In [2]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Use the structured data partitions created above
print("Initializing GPT-2 workflow...")

# 2. Tokenize using GPT-2
gpt_checkpoint = "gpt2"
gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_checkpoint)

# CRITICAL GPT ADJUSTMENT: GPT models don't have a native padding token. We force it to use end-of-text token.
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

def tokenize_gpt(examples):
    text_list = [str(t) for t in examples["text"]]
    return gpt_tokenizer(text_list, truncation=True, padding="max_length", max_length=64)

tokenized_train_gpt = train_dataset.map(tokenize_gpt, batched=True)
tokenized_eval_gpt = eval_dataset.map(tokenize_gpt, batched=True)

# 3. Initialize GPT-2 for Classification
# This adds a linear layer at the end of the causal transformer structure
gpt_model = AutoModelForSequenceClassification.from_pretrained(gpt_checkpoint, num_labels=2)
gpt_model.config.pad_token_id = gpt_tokenizer.pad_token_id

# 4. Training Arguments
gpt_args = TrainingArguments(
    output_dir="./gpt_results",
    eval_strategy="epoch",
    learning_rate=3e-5, # GPT models often favor slightly higher initialization rates than BERT
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

gpt_trainer = Trainer(
    model=gpt_model,
    args=gpt_args,
    train_dataset=tokenized_train_gpt,
    eval_dataset=tokenized_eval_gpt,
    compute_metrics=compute_metrics,
)

print("\nTraining GPT-2...")
gpt_trainer.train()

print("\nEvaluating GPT-2 metrics...")
gpt_eval = gpt_trainer.evaluate()

print("\n--- RESULTS FOR ASSIGNMENT 2 ---")
print(f"GPT-2 Test Accuracy: {gpt_eval['eval_accuracy']:.4f}")
print(f"GPT-2 Test F1-Score: {gpt_eval['eval_f1']:.4f}")

Initializing GPT-2 workflow...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Evaluating GPT-2 metrics...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
1.033322,1.107054,2,0.640000,0.619465



--- RESULTS FOR ASSIGNMENT 2 ---
GPT-2 Test Accuracy: 0.6400
GPT-2 Test F1-Score: 0.6195
